In [ ]:
import pandas as pd
import numpy as np
from datetime import date

pd.set_option("display.max_columns", None)
# df = pd.read_csv("../data/train.csv")
df = pd.read_csv("train.csv")
df = df.drop(columns=['Unnamed: 0'])
print(df.shape)
df.head()

(5847, 13)


,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5847 entries, 0 to 5846
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               5847 non-null   object 
 1   Location           5847 non-null   object 
 2   Year               5847 non-null   int64  
 3   Kilometers_Driven  5847 non-null   int64  
 4   Fuel_Type          5847 non-null   object 
 5   Transmission       5847 non-null   object 
 6   Owner_Type         5847 non-null   object 
 7   Mileage            5845 non-null   object 
 8   Engine             5811 non-null   object 
 9   Power              5811 non-null   object 
 10  Seats              5809 non-null   float64
 11  New_Price          815 non-null    object 
 12  Price              5847 non-null   float64
dtypes: float64(2), int64(2), object(9)
memory usage: 594.0+ KB


A. Missing Values
1. Find


In [ ]:
missing = pd.DataFrame({'missing_count': df.isna().sum(),
                        'missing_pct': (df.isna().mean() * 100).round(2)})
missing[missing.missing_count > 0].sort_values('missing_pct', ascending=False)

,missing_count,missing_pct
New_Price,5032,86.06
Seats,38,0.65
Power,36,0.62
Engine,36,0.62
Mileage,2,0.03


In [ ]:
print('Mileage recorded as 0:', (df['Mileage'].str.split().str[0].astype(float) == 0).sum())

Mileage recorded as 0: 40


2. Drop or impute?


New_Price(~86%) -- **Drop the column** (after cleaning it in part b)  With 5 out of 6 values missing, any mean/median imputation would invent most of the column and give a near-constant, misleading feature.

Engine, Power(0.6%) -- **Drop the rows** Very few rows; these are the same cars (missing engine specs together), so dropping loses <1% of the data with no bias risk. Imputing a single median engine size for an Audi and a Maruti alike would be inaccurate.

Seats(0.65%) -- **Drop the rows**  Mostly the same rows as Engine/Power – tiny loss.

Mileage (NaN) 2 rows -- **Drop the rows**  Negligible.

Mileage = 0 | ~40 rows -- **Impute with the median** (done in part b once Mileage is numeric) Median rather than mean because mileage is skewed and the median is robust to outliers.

In [ ]:
before = len(df)
df = df.dropna(subset=['Mileage', 'Engine', 'Power', 'Seats'])
print(f'Rows before: {before}, after: {len(df)}, dropped: {before - len(df)}')
df.isna().sum()

Rows before: 5847, after: 5807, dropped: 40


,0
Name,0
Location,0
Year,0
Kilometers_Driven,0
Fuel_Type,0
Transmission,0
Owner_Type,0
Mileage,0
Engine,0
Power,0


b. Remove units, keep only numbers

- Mileage: remove kmpl / km/kg
- Engine: remove CC
- Power: remove bhp
- New_Price: remove Lakh. A few values are in Cr (crore); 1 crore = 100 lakh, so those are multiplied by 100 to keep every price in lakhs.

In [ ]:
df['Mileage'] = df['Mileage'].str.split().str[0].astype(float)
df['Engine']  = df['Engine'].str.replace(' CC', '', regex=False).astype(float)
df['Power']   = pd.to_numeric(df['Power'].str.replace(' bhp', '', regex=False), errors='coerce')

def to_lakh(x):
    if pd.isna(x):
        return np.nan
    value, unit = x.split()
    value = float(value)
    return value * 100 if unit.lower().startswith('cr') else value

df['New_Price'] = df['New_Price'].apply(to_lakh)
df[['Mileage', 'Engine', 'Power', 'New_Price']].dtypes

,0
Mileage,float64
Engine,float64
Power,float64
New_Price,float64


In [ ]:
df[['Mileage', 'Engine', 'Power', 'New_Price']].head()

,Mileage,Engine,Power,New_Price
0,19.67,1582.0,126.20,NaN
1,13.00,1199.0,88.70,8.61
2,20.77,1248.0,88.76,NaN
3,15.20,1968.0,140.80,NaN
4,23.08,1461.0,63.10,NaN


- Replace impossible Mileage == 0 with the median mileage.
- Drop New_Price (86% missing – see justification above).

In [ ]:
median_mileage = df.loc[df['Mileage'] > 0, 'Mileage'].median()
print('Median mileage used for imputation:', median_mileage)
df.loc[df['Mileage'] == 0, 'Mileage'] = median_mileage

df = df.drop(columns=['New_Price'])
print('Remaining missing values:', df.isna().sum().sum())
df.shape

Median mileage used for imputation: 18.2
Remaining missing values: 0


(5807, 12)

c. One-hot encode Fuel_Type and Transmission

In [ ]:
print(df['Fuel_Type'].value_counts(), '\n')
print(df['Transmission'].value_counts())

Fuel_Type
Diesel    3152
Petrol    2655
Name: count, dtype: int64 

Transmission
Manual       4105
Automatic    1702
Name: count, dtype: int64


In [ ]:
df = pd.get_dummies(df, columns=['Fuel_Type', 'Transmission'], dtype=int)
df.filter(like='Fuel_Type').join(df.filter(like='Transmission')).head()

,Fuel_Type_Diesel,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual
0,1,0,0,1
1,0,1,0,1
2,1,0,0,1
3,1,0,1,0
4,1,0,0,1


d. Create a new feature – Car_Age

In [ ]:
current_year = pd.Timestamp.now().year
df['Car_Age'] = current_year - df['Year']
df[['Name', 'Year', 'Car_Age']].head()

,Name,Year,Car_Age
0,Hyundai Creta 1.6 CRDi SX Option,2015,11
1,Honda Jazz V,2011,15
2,Maruti Ertiga VDI,2012,14
3,Audi A4 New 2.0 TDI Multitronic,2013,13
4,Nissan Micra Diesel XV,2013,13


e. dplyr-style operations in pandas

select – keep only a few columns

In [ ]:
selected = df[['Name', 'Location', 'Year', 'Kilometers_Driven', 'Mileage', 'Power', 'Car_Age', 'Price']]
selected.head()

,Name,Location,Year,Kilometers_Driven,Mileage,Power,Car_Age,Price
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,19.67,126.20,11,12.50
1,Honda Jazz V,Chennai,2011,46000,13.00,88.70,15,4.50
2,Maruti Ertiga VDI,Chennai,2012,87000,20.77,88.76,14,6.00
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,15.20,140.80,13,17.74
4,Nissan Micra Diesel XV,Jaipur,2013,86999,23.08,63.10,13,3.50


filter – automatic diesel cars up to 10 years old priced above 10 lakh

In [ ]:
filtered = df[(df['Transmission_Automatic'] == 1) & (df['Fuel_Type_Diesel'] == 1)
              & (df['Car_Age'] <= 10) & (df['Price'] > 10)]
print(len(filtered), 'cars match')
filtered[['Name', 'Year', 'Price']].head()

328 cars match


,Name,Year,Price
5,Toyota Innova Crysta 2.8 GX AT 8S,2016,17.50
65,Mercedes-Benz C-Class Progressive C 220d,2019,35.67
87,Audi A6 2011-2015 35 TDI Premium,2018,39.58
105,Toyota Innova Crysta 2.8 ZX AT,2017,19.25
124,Mahindra XUV500 AT W10 FWD,2016,13.25


rename – clearer column names

In [ ]:
renamed = df.rename(columns={'Kilometers_Driven': 'Km_Driven',
                             'Price': 'Price_Lakh',
                             'Power': 'Power_bhp',
                             'Engine': 'Engine_CC'})
renamed.columns.tolist()

['Name',
 'Location',
 'Year',
 'Km_Driven',
 'Owner_Type',
 'Mileage',
 'Engine_CC',
 'Power_bhp',
 'Seats',
 'Price_Lakh',
 'Fuel_Type_Diesel',
 'Fuel_Type_Petrol',
 'Transmission_Automatic',
 'Transmission_Manual',
 'Car_Age']

mutate – add Brand and kilometres driven per year

In [ ]:
df = df.assign(
    Brand=(df['Name'].str.split().str[0]
             .replace({'Land': 'Land Rover', 'ISUZU': 'Isuzu'})),
    Km_per_Year=(df['Kilometers_Driven'] / df['Car_Age'].clip(lower=1)).round(0)
)
df[['Name', 'Brand', 'Kilometers_Driven', 'Car_Age', 'Km_per_Year']].head()

,Name,Brand,Kilometers_Driven,Car_Age,Km_per_Year
0,Hyundai Creta 1.6 CRDi SX Option,Hyundai,41000,11,3727.0
1,Honda Jazz V,Honda,46000,15,3067.0
2,Maruti Ertiga VDI,Maruti,87000,14,6214.0
3,Audi A4 New 2.0 TDI Multitronic,Audi,40670,13,3128.0
4,Nissan Micra Diesel XV,Nissan,86999,13,6692.0


arrange – most expensive cars first

In [ ]:
df.sort_values('Price', ascending=False)[['Name', 'Year', 'Location', 'Price']].head(10)

,Name,Year,Location,Price
3952,Land Rover Range Rover 3.0 Diesel LWB Vogue,2017,Hyderabad,160.00
5620,Lamborghini Gallardo Coupe,2011,Delhi,120.00
5752,Jaguar F Type 5.0 V8 S,2015,Hyderabad,100.00
1457,Land Rover Range Rover Sport SE,2019,Kochi,97.07
1917,BMW 7 Series 740Li,2018,Coimbatore,93.67
1927,BMW 7 Series 740Li,2017,Bangalore,93.00
4556,Mercedes-Benz SLK-Class 55 AMG,2014,Bangalore,90.00
5380,BMW X6 xDrive 40d M Sport,2015,Ahmedabad,85.00
2035,Mercedes-Benz SLC 43 AMG,2019,Coimbatore,83.96
2347,Jaguar XJ 3.0L Portfolio,2016,Delhi,79.00


group_by + summarize – statistics by Brand

In [ ]:
brand_summary = (df.groupby('Brand')
                   .agg(n_cars=('Price', 'size'),
                        avg_price=('Price', 'mean'),
                        median_price=('Price', 'median'),
                        avg_age=('Car_Age', 'mean'),
                        avg_power=('Power', 'mean'))
                   .round(2)
                   .sort_values('avg_price', ascending=False))
brand_summary.head(10)

,n_cars,avg_price,median_price,avg_age,avg_power
Brand,,,,,
Lamborghini,1,120.00,120.00,15.00,560.00
Bentley,1,59.00,59.00,20.00,552.00
Porsche,16,49.20,49.36,12.44,331.29
Land Rover,57,39.86,35.00,11.58,189.60
Jaguar,40,37.63,31.90,11.60,239.08
Mercedes-Benz,316,26.92,24.23,12.40,193.58
Mini,26,26.90,24.28,10.92,147.44
Audi,235,25.57,23.50,12.38,192.11
BMW,262,25.44,21.42,12.92,209.82


group_by + summarize – by Location

In [ ]:
(df.groupby('Location')
   .agg(n_cars=('Price', 'size'), avg_price=('Price', 'mean'), avg_km=('Kilometers_Driven', 'mean'))
   .round(2)
   .sort_values('avg_price', ascending=False))

,n_cars,avg_price,avg_km
Location,,,
Coimbatore,629,15.14,46906.34
Bangalore,347,13.64,58172.25
Kochi,637,11.32,44695.48
Hyderabad,709,10.00,70680.36
Delhi,535,9.91,57099.69
Mumbai,757,9.57,44551.67
Ahmedabad,217,8.59,54951.52
Chennai,474,7.96,90564.88
Pune,581,6.99,69749.08


Chaining everything together (like a dplyr pipe %>%)

In [ ]:
(df[['Brand', 'Owner_Type', 'Transmission_Automatic', 'Kilometers_Driven', 'Price']]
   .query('Kilometers_Driven < 100000')
   .rename(columns={'Transmission_Automatic': 'Is_Automatic'})
   .assign(Price_Rupees=lambda d: d['Price'] * 100000)
   .groupby('Owner_Type')
   .agg(n=('Price', 'size'), avg_price_lakh=('Price', 'mean'),
        pct_automatic=('Is_Automatic', 'mean'))
   .round(3)
   .sort_values('avg_price_lakh', ascending=False))

,n,avg_price_lakh,pct_automatic
Owner_Type,,,
First,4444,10.299,0.298
Second,767,8.323,0.338
Third,69,5.777,0.246
Fourth & Above,5,3.174,0.200


In [ ]:
df.to_csv('train_cleaned.csv', index=False)
print('Saved cleaned data:', df.shape)

Saved cleaned data: (5807, 17)
